In [21]:
from dotenv import load_dotenv
from groq import Groq
import os, json, hashlib
import fitz
from IPython.display import display, Markdown 
load_dotenv()





True

### Initialise the LLM

In [22]:
def init_llm(groq_api_key: str): 
    print("Groq key loaded:", "✅" if groq_api_key else "❌ Missing!")
    client = Groq(api_key=groq_api_key)
    print("✅ Groq client ready")
    return client

GROQ_KEY    = os.getenv("GROQ_API_KEY")
groq_client = init_llm(GROQ_KEY)

Groq key loaded: ✅
✅ Groq client ready


### Local Tree Creation

In [23]:


TREE_CACHE_DIR = "./tree_cache"
os.makedirs(TREE_CACHE_DIR, exist_ok=True)

def get_file_hash(file_path: str) -> str:
    """Unique fingerprint of a file based on its contents."""
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

def build_local_tree(pdf_path: str) -> list:   # this function creates a local copy of the tree for better performance. It's required because pageindex has rate limits.
    """
    Parse a PDF into a structured node tree locally using PyMuPDF.
    Each page becomes one node — title, page number, and full text.

    """
    doc  = fitz.open(pdf_path)
    tree = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if not text:
            continue  # skip blank pages
        tree.append({
            "node_id":    f"{i+1:04d}",
            "title":      f"Page {i+1}",
            "page_index": i + 1,
            "text":       text,
            "nodes":      []   # no children — flat page-level tree
        })
    doc.close()
    return tree

def get_or_build_tree(pdf_path: str) -> list:
    """
    Build tree once, cache to disk, reuse forever.
    Same file → same hash → instant load.
    Works across server restarts and multiple users.
    
    """
    file_hash  = get_file_hash(pdf_path)
    cache_path = os.path.join(TREE_CACHE_DIR, f"{file_hash}.json")

    if os.path.exists(cache_path):
        print("✅ Tree found in cache — loading instantly")
        with open(cache_path, "r") as f:
            return json.load(f)

    print(f"🔨 New document — building tree for {pdf_path}...")
    tree = build_local_tree(pdf_path)

    with open(cache_path, "w") as f:
        json.dump(tree, f, indent=2)

    print(f"✅ Tree built and cached ({len(tree)} pages)")
    return tree

### Tree Inspection

In [24]:

def print_tree(nodes: list, indent: int = 0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

def count_nodes(nodes: list) -> int:
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

### LLM Tree Search

In [25]:

def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    """
    Sends the query + compressed document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    Returns: dict with 'thinking' and 'node_list'.

    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are a precise document navigation assistant. Given a query and a document's hierarchical tree structure (representing its Table of Contents with node IDs), your job is to identify the exact node IDs whose sections are most likely to contain the answer.

## Rules
- Select ONLY nodes whose content would directly answer the query — not parent/ancestor nodes unless no specific child exists.
- Prefer the most specific (deepest) relevant node over a broad parent node.
- If multiple sibling nodes are relevant, include all of them.
- If NO node is relevant, return an empty list — do not guess.
- Do NOT include nodes based on superficial keyword overlap alone; reason about actual content relevance.
- Return a maximum of 5 node IDs. Prioritize precision over recall.

## Query
{query}

## Document Tree
{json.dumps(compressed_tree, indent=2)}

## Output Format
Reply ONLY with valid JSON. No explanation outside the JSON block. No markdown fences. No extra keys.

{{
  "thinking": "<step-by-step reasoning: what the query is asking, which sections address it, why you included or excluded specific nodes>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Node Retrieval

In [26]:

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

### Answer Generation

In [31]:

def generate_answer(query: str, nodes: list, model: str = "llama-3.3-70b-versatile") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Returns a plain string — rendering is handled by the caller.
    
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    system_message = """You are an expert document analyst with strict grounding discipline.

## Core Rules
- Answer ONLY using information explicitly present in the provided context.
- You MUST use ALL provided context sections in your answer, not just the first one.
- If the context does not contain enough information to answer, respond exactly with:
  "The provided context does not contain sufficient information to answer this question."
- Never infer, assume, or use outside knowledge — even if you are confident.
- Never fabricate citations, page numbers, or section titles.
- If the context contains mathematical formulas, you must include them in your response,else include them in your answer.
- Format all formulas using standard LaTeX enclosed in $$ for block equations and $ for inline equations.

## Citation Format
After every distinct claim, cite inline as: (Section: "<title>", Page: <number>)

## Answer Quality
- Lead with the answer directly and explain a little in points — no preamble.
- Show politeness and normal interactive behaviour.
- Use bullet points for multi-part answers; prose for single-fact answers.
- Structure with subheadings if multiple sections cover distinct subtopics.
- Wrap all equations in $$...$$ for block display."""

    user_message = f"""## Question
{query}

## Context
{context}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )

    return response.choices[0].message.content

### VecLess RAG

In [32]:


def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end Vectorless RAG pipeline:
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content from tree
    Step 3: Answer Generation → produces cited, grounded answer
    
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree search
    search_result = llm_tree_search(query, tree)
    node_ids      = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    # Single render point
    display(Markdown(answer))

    return answer

### Testing

In [33]:

PDF_PATH = "../../sample_document.pdf"

# Build or load tree (runs once, cached forever)
pageindex_tree = get_or_build_tree(PDF_PATH)

# Inspect structure
print("\n📚 Document Structure:\n")
print_tree(pageindex_tree)
print(f"\n🔢 Total nodes: {count_nodes(pageindex_tree)}")



✅ Tree found in cache — loading instantly

📚 Document Structure:

[0001] Page 1  (p.1)
[0002] Page 2  (p.2)
[0003] Page 3  (p.3)
[0004] Page 4  (p.4)
[0005] Page 5  (p.5)
[0006] Page 6  (p.6)
[0007] Page 7  (p.7)
[0008] Page 8  (p.8)
[0009] Page 9  (p.9)
[0010] Page 10  (p.10)
[0011] Page 11  (p.11)
[0012] Page 12  (p.12)
[0013] Page 13  (p.13)
[0014] Page 14  (p.14)
[0015] Page 15  (p.15)
[0016] Page 16  (p.16)
[0017] Page 17  (p.17)
[0018] Page 18  (p.18)
[0019] Page 19  (p.19)
[0020] Page 20  (p.20)
[0021] Page 21  (p.21)
[0022] Page 22  (p.22)
[0023] Page 23  (p.23)
[0024] Page 24  (p.24)
[0025] Page 25  (p.25)
[0026] Page 26  (p.26)
[0027] Page 27  (p.27)
[0028] Page 28  (p.28)
[0029] Page 29  (p.29)
[0030] Page 30  (p.30)
[0031] Page 31  (p.31)
[0032] Page 32  (p.32)
[0033] Page 33  (p.33)
[0034] Page 34  (p.34)
[0035] Page 35  (p.35)
[0036] Page 36  (p.36)
[0037] Page 37  (p.37)
[0038] Page 38  (p.38)
[0039] Page 39  (p.39)
[0040] Page 40  (p.40)
[0041] Page 41  (p.41)
[0042] Pa

In [34]:
# Run a query
answer = vectorless_rag(
    query="list down the main formulas in the pdf?",
    tree=pageindex_tree
)

🔍 Query: list down the main formulas in the pdf?

🧠 Reasoning: The query is asking for the main formulas in the PDF, which suggests that we are looking for sections that explicitly state formulas or equations used in Principal Component Analysis (PCA) or possibly...
🎯 Retrieved node IDs: ['0006', '0007', '0008', '0033', '0035']
📄 Sections found: ['Page 6', 'Page 7', 'Page 8', 'Page 33', 'Page 35']


The main formulas are not explicitly listed in the provided context. However, some mathematical concepts are mentioned:
* Linear transformation implied by PCA (Section: "Page 8", Page 8)
* Linear transformation implied by LDA, which involves the eigenvectors of $S_w^{-1} S_b$ (Section: "Page 35", Page 35)
 
Unfortunately, the provided context does not contain sufficient information to list down the main formulas.